# 惯性导航 + 地磁融合（在线试运行）

**打开方式：** [Google Colab](https://colab.research.google.com/) → **文件 → 上传笔记本** → 选择本机 `INS_Mag_Nav_Colab.ipynb`

或把本笔记本拖进 Colab 页面。运行顺序：**从上到下依次 Shift+Enter**。

In [ ]:
!pip install -q numpy matplotlib

## 1. 融合算法（单格复制即可运行）

In [ ]:
import numpy as np

def normalize(v, eps=1e-12):
    n = np.linalg.norm(v)
    return v / n if n >= eps else np.array([0., 0., 1.])

def skew(v):
    x, y, z = v
    return np.array([[0,-z,y],[z,0,-x],[-y,x,0]], float)

def quat_multiply(q, p):
    w1,x1,y1,z1 = q; w2,x2,y2,z2 = p
    return np.array([w1*w2-x1*x2-y1*y2-z1*z2, w1*x2+x1*w2+y1*z2-z1*y2,
                     w1*y2-x1*z2+y1*w2+z1*x2, w1*z2+x1*y2-y1*x2+z1*w2])

def quat_normalize(q):
    n = np.linalg.norm(q)
    return q/n if n > 1e-12 else np.array([1.,0,0,0])

def quat_from_delta_angle(dtheta):
    half = 0.5*dtheta; th = np.linalg.norm(half)
    if th < 1e-12: return np.array([1.,0,0,0])
    s = np.sin(th)/th
    return quat_normalize(np.array([np.cos(th), half[0]*s, half[1]*s, half[2]*s]))

def quat_to_rot_b_to_n(q):
    w,x,y,z = quat_normalize(q)
    return np.array([
        [1-2*(y*y+z*z), 2*(x*y-w*z), 2*(x*z+w*y)],
        [2*(x*y+w*z), 1-2*(x*x+z*z), 2*(y*z-w*x)],
        [2*(x*z-w*y), 2*(y*z+w*x), 1-2*(x*x+y*y)]])

def integrate_quat_gyro(q, gyro, dt):
    w = np.linalg.norm(gyro)
    if w < 1e-12: return quat_normalize(q)
    ha = 0.5*w*dt; s = np.sin(ha)/w
    dq = np.array([np.cos(ha), gyro[0]*s, gyro[1]*s, gyro[2]*s])
    return quat_normalize(quat_multiply(q, dq))

class MEKF9:
    def __init__(self, dt, m_unit_n, sigma_gyro=0.02, sigma_mag=0.05, sigma_accel=0.3):
        self.dt = dt
        self.g_unit_n = np.array([0.,0.,1.])
        self.m_unit_n = normalize(np.asarray(m_unit_n, float))
        self.q = np.array([1.,0,0,0])
        self.x = np.zeros(9)
        self.P = np.eye(9)*0.01
        self.Q = np.zeros((9,9))
        self.Q[:3,:3] = np.eye(3)*(sigma_gyro**2)*dt
        self.Q[3:6,3:6] = np.eye(3)*1e-8*dt
        self.Q[6:,6:] = np.eye(3)*1e-8*dt
        self.R_acc = np.eye(3)*(sigma_accel**2)
        self.R_mag = np.eye(3)*(sigma_mag**2)

    def predict(self, gyro, accel=None):
        omega = gyro - self.x[3:6]
        self.q = integrate_quat_gyro(self.q, omega, self.dt)
        F = np.eye(9); F[:3,:3] -= skew(omega)*self.dt; F[:3,3:6] = -np.eye(3)*self.dt
        self.P = F@self.P@F.T + self.Q

    def _inject(self):
        self.q = quat_normalize(quat_multiply(self.q, quat_from_delta_angle(self.x[:3])))
        self.x[:3] = 0

    def _update(self, z, h, H, R):
        y = z-h; S = H@self.P@H.T + R; K = self.P@H.T@np.linalg.inv(S)
        self.x += K@y; I = np.eye(9)
        self.P = (I-K@H)@self.P@(I-K@H).T + K@R@K.T; self._inject()

    def update_accel(self, accel):
        C = quat_to_rot_b_to_n(self.q)
        h = normalize(C.T @ (-self.g_unit_n))
        z = normalize(accel)
        H = np.zeros((3,9)); H[:3,:3]=skew(h); H[:3,6:]=np.eye(3)
        self._update(z,h,H,self.R_acc)

    def update_mag(self, mag):
        C = quat_to_rot_b_to_n(self.q)
        h = normalize(C.T @ self.m_unit_n)
        z = normalize(mag)
        H = np.zeros((3,9)); H[:3,:3]=skew(h)
        self._update(z,h,H,self.R_mag)

class InsMagNavigator:
    def __init__(self, dt, mag_field_ned):
        self.dt = dt
        self.ekf = MEKF9(dt, mag_field_ned)
        self.p = np.zeros(3); self.v = np.zeros(3); self.g = np.array([0.,0.,9.81])

    def step_imu(self, gyro, accel, use_acc=True):
        self.ekf.predict(gyro, accel)
        if use_acc: self.ekf.update_accel(accel)
        C = quat_to_rot_b_to_n(self.ekf.q)
        a = C @ (accel - self.ekf.x[6:9]) + self.g
        self.v += a*self.dt; self.p += self.v*self.dt

    def step_mag(self, mag):
        self.ekf.update_mag(mag)

    def yaw_deg(self):
        C = quat_to_rot_b_to_n(self.ekf.q)
        return np.rad2deg(np.arctan2(C[1,0], C[0,0]))

print('OK: navigator ready')

## 2. 三个内置数据例子

| 例子 | 含义 | 预期现象 |
|------|------|----------|
| `DATA_STATIC` | 静止 2s | 航向基本不变，位置接近 0 |
| `DATA_TURN` | 转弯约 10s | 航向持续变化（约 30°） |
| `DATA_FORWARD` | 加速+微转 6s | 北向速度/位置增加 |

In [ ]:
DT = 0.01
MAG_N = np.array([np.cos(np.deg2rad(60)), 0, np.sin(np.deg2rad(60))])
G = np.array([0.,0.,9.81])
rng = np.random.default_rng(0)

def synth_motion(seconds, turn_rate_deg_s=0., forward_accel=0., noise=False):
    """返回 list of (gyro, accel, mag)"""
    q = np.array([1.,0,0,0])
    tr = np.deg2rad(turn_rate_deg_s)
    out = []
    for _ in range(int(seconds/DT)):
        gyro = np.array([0.,0.,tr])
        C = quat_to_rot_b_to_n(q)
        accel = C.T @ (np.array([forward_accel,0,0]) - G)
        mag = C.T @ MAG_N; mag = mag/np.linalg.norm(mag)*40
        if noise:
            gyro += rng.normal(0,0.002,3)
            accel += rng.normal(0,0.03,3)
            mag += rng.normal(0,0.5,3)
        out.append((gyro.copy(), accel.copy(), mag.copy()))
        q = integrate_quat_gyro(q, gyro, DT)
    return out

DATA_STATIC = [(np.zeros(3), np.array([0,0,-9.81]), np.array([20.,0.,35.])) for _ in range(200)]
DATA_TURN = synth_motion(10, turn_rate_deg_s=3, noise=True)
DATA_FORWARD = synth_motion(6, turn_rate_deg_s=0.5, forward_accel=0.3, noise=True)
print('static', len(DATA_STATIC), 'turn', len(DATA_TURN), 'forward', len(DATA_FORWARD))

In [ ]:
def run_dataset(name, samples, mag_every=5):
    nav = InsMagNavigator(DT, MAG_N)
    yaws, pos = [], []
    for k, (g,a,m) in enumerate(samples):
        nav.step_imu(g, a, use_acc=(k%2==0))
        if k % mag_every == 0:
            nav.step_mag(m)
        yaws.append(nav.yaw_deg())
        pos.append(nav.p.copy())
    pos = np.array(pos); yaws = np.array(yaws)
    print('---', name, '---')
    print('  时长 %.1f s, 样本 %d' % (len(samples)*DT, len(samples)))
    print('  末位置 NED (m):', np.round(pos[-1], 2))
    print('  航向变化 (deg): %.2f' % (yaws[-1]-yaws[0]))
    return pos, yaws

run_dataset('静止', DATA_STATIC)
run_dataset('转弯', DATA_TURN)
run_dataset('加速', DATA_FORWARD)

## 3. 画图（转弯例子）

In [ ]:
import matplotlib.pyplot as plt
pos, yaws = run_dataset('转弯(绘图)', DATA_TURN)
t = np.arange(len(yaws))*DT
fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].plot(t, pos[:,0], label='North'); ax[0].plot(t, pos[:,1], label='East')
ax[0].set_xlabel('t (s)'); ax[0].set_ylabel('m'); ax[0].legend(); ax[0].set_title('Position')
ax[1].plot(t, yaws); ax[1].set_xlabel('t (s)'); ax[1].set_ylabel('yaw (deg)')
plt.tight_layout(); plt.show()

## 4. 上传你自己的 CSV

列名：`gx,gy,gz,ax,ay,az,mx,my,mz`（陀螺 rad/s，加计 m/s²，磁力计任意单位）

In [ ]:
from google.colab import files
import io, csv

uploaded = files.upload()  # 选你的 csv
fname = list(uploaded.keys())[0]
rows = []
for line in uploaded[fname].decode().splitlines():
    line = line.strip()
    if not line or line.startswith('#') or line.lower().startswith('gx'): continue
    rows.append(list(map(float, line.split(',')[:9])))

nav = InsMagNavigator(DT, MAG_N)
for k, r in enumerate(rows):
    nav.step_imu(np.array(r[:3]), np.array(r[3:6]), use_acc=True)
    if k%5==0: nav.step_mag(np.array(r[6:9]))
print('上传文件', fname, '行数', len(rows), '末航向(deg)', round(nav.yaw_deg(),2))